In [1]:
import numpy as np
import pandas as pd
import os 
import glob
import chardet

In [14]:
os.getcwd()

'c:\\Users\\Prithwijit\\Dropbox\\My PC (LAPTOP-7SET5QFO)\\Documents\\jupyternotebook_ws\\Database_code\\campaign03\\xrf'

In [16]:
os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir))

# KT20 cond and mag

In [5]:
def cond_sus_proc(drillhole,prop):

    data_inventory = pd.read_excel(f'{drillhole}/{drillhole}_sample_report.xlsx', sheet_name = None)
    df_record = data_inventory.get(prop)

    os.chdir(f'{drillhole}/cond_sus')
    

    dic_result = {'sample id':[],'mean':[], 'std':[],'median':[], 'max':[],'min':[]} ## creating dataframe for results #######


    for dir in os.listdir():
        os.chdir(dir+'/'+'kt20_0385')

        try:
            meta_data = pd.read_csv('kappa_sigma_0385.csv')
            df_record_sub = df_record[df_record['measurement_id'].isin(meta_data['Record_Id'])]
            
        except KeyError:
            meta_data = pd.read_csv('kappa_sigma_0385.csv',sep=';')
            df_record_sub = df_record[df_record['measurement_id'].isin(meta_data['Record_Id'])]

                                  
        record_id_list_sub = df_record_sub['measurement_id'].tolist()
        record_id_strings_sub = [str(record_id) for record_id in record_id_list_sub] 
        sample_id_sub = df_record_sub['sample_id'].tolist()
        sample_id_strings_sub = [str(sample_id) for sample_id in sample_id_sub]

        for i in range(len(record_id_strings_sub)):
            os.chdir(str(record_id_strings_sub[i]))   #### changing directory iteratively##########
            file = glob.glob('*.csv')
            df = pd.read_csv(file[0])    
            a1 = df.index[df['Date'] == 'Min'].to_list()
            a1 = int(np.array(a1))
            d1 = np.array(df.iloc[2:a1,1])
            d1 = d1.astype('float32')
            d1 = np.reshape(d1, (a1-2,1))
            w1 = np.ma.masked_outside(d1,np.mean(d1)+(2*np.std(d1)),np.mean(d1)-(2*np.std(d1)))
            dic_result['sample id'].append(sample_id_strings_sub[i]),dic_result['mean'].append(np.mean(w1)),dic_result['std'].append(np.std(w1)),dic_result['median'].append(np.median(w1)),dic_result['max'].append(np.max(w1)),dic_result['min'].append(np.min(w1))
            os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir))
            # print(os.getcwd())
        os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir+os.sep + os.pardir))

    os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir + os.sep + os.pardir))

    df_proc = pd.DataFrame(dic_result)

    sample_id = df_record['sample_id']
    nan_indices = df_record[df_record['measurement_id'].isna()].index

    dic_nan = {str(i) : ['nan'] for i in df_proc.columns}
    df_nan = pd.DataFrame(dic_nan)

    
    for index in nan_indices:
        df_proc = pd.concat([df_proc.iloc[:index], df_nan, df_proc.iloc[index:]]).reset_index(drop=True)
    result_df = df_proc.reset_index(drop=True)

    result_df.iloc[:,0] = (sample_id).astype(str)
    cal = result_df['sample id'].str.contains('cal')
    df_cal = result_df[cal]
    df_final = result_df[np.logical_not(cal)]
    df_final.index = range(len(df_final))

    return df_final,df_cal

In [6]:
_,t = cond_sus_proc('TC-2531-001','sus')

c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-packages\numpy\core\fromnumeric.py:758: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-packages\numpy\core\fromnumeric.py:758: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-packages\numpy\core\fromnumeric.py:758: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-packages\numpy\core\fromnumeric.py:758: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-p

In [7]:
t

,sample id,mean,std,median,max,min
0,cal_001,96.952413,1.460755,97.316696,98.698097,80.524597
11,cal_002,97.802106,0.539197,98.087601,98.204201,94.962700
22,cal_003,96.572523,1.207703,96.816803,97.320999,86.658203
33,cal_004,96.524447,0.672751,96.603104,97.257301,88.355202
44,cal_005,97.255727,0.505718,97.083801,98.015999,95.102097
55,cal_006,97.743068,0.921557,97.801697,99.108902,85.997597
66,cal_007,97.136983,0.863204,97.273003,97.323196,90.176399
77,cal_008,97.451911,0.388149,97.358902,98.085503,93.925598
83,cal_009,97.418092,0.200519,97.4571,97.593803,95.937500
94,cal_010,97.675100,0.491712,97.538895,99.067802,96.884399


# Gamma processing

In [17]:
def gamma(drillhole):     
    
    dict = pd.read_excel(f'{drillhole}/{drillhole}_sample_report.xlsx', sheet_name = None)
    df_record = dict.get('gamma') 
    
    os.chdir(f'{drillhole}/gamma')

    # measurement_id_array = np.c_[df_record['measurement_id1'],df_record['measurement_id2'],df_record['measurement_id3']]
    dic_result = {'sample id':[],'K_mean':[],'K_median':[], 'K_max':[],'K_min':[], 'K_std':[],'U_mean':[],'U_median':[], 
                  'U_max':[],'U_min':[], 'U_std':[],'Th_mean':[],'Th_median':[], 'Th_max':[],'Th_min':[], 'Th_std':[]}

    for dir in os.listdir():
        os.chdir(f'{dir}/gt40_0154/assay')

        # file_record = glob.glob('*.xlsx')
        # df_record = pd.read_excel(file_record[0])

        file_assay = glob.glob('*.csv')
        assay = pd.read_csv(file_assay[0],sep=';')
        df_assay = assay[assay['Id'].notna()]
        df_record_sub = df_record[df_record['measurement_id1'].isin(df_assay['Id']) + 
                                  df_record['measurement_id2'].isin(df_assay['Id']) + df_record['measurement_id3'].isin(df_assay['Id'])]
        sampleid = np.array(df_record_sub['sample_id'])
        measurement_id_array = np.c_[df_record_sub['measurement_id1'],df_record_sub['measurement_id2'],df_record_sub['measurement_id3']] 

        
        for i in range (len(df_record_sub)):
            measurement_id_temp = measurement_id_array[i,:].tolist()
            df_temp = df_assay[df_assay['Id'].isin(measurement_id_temp)]
            df_temp.replace('<  0.00000', '0', inplace=True)
            dic_result['sample id'].append(str(sampleid[i])),dic_result['K_mean'].append(df_temp['Component_1 value'].astype('float16').mean()),
            dic_result['K_median'].append(df_temp['Component_1 value'].astype('float16').median()),dic_result['K_max'].append(df_temp['Component_1 value'].astype('float16').max()),
            dic_result['K_min'].append(df_temp['Component_1 value'].astype('float16').min()),dic_result['K_std'].append(df_temp['Component_1 value'].astype('float16').std()),
            dic_result['U_mean'].append(df_temp['Component_2 value'].astype('float16').mean()),dic_result['U_median'].append(df_temp['Component_2 value'].astype('float16').median()),
            dic_result['U_max'].append(df_temp['Component_2 value'].astype('float16').max()),dic_result['U_min'].append(df_temp['Component_2 value'].astype('float16').min()),
            dic_result['U_std'].append(df_temp['Component_2 value'].astype('float16').std()),
            dic_result['Th_mean'].append(df_temp['Component_3 value'].astype('float16').mean()),dic_result['Th_median'].append(df_temp['Component_3 value'].astype('float16').median()),
            dic_result['Th_max'].append(df_temp['Component_3 value'].astype('float16').max()),dic_result['Th_min'].append(df_temp['Component_3 value'].astype('float16').min()),dic_result['Th_std'].append(df_temp['Component_3 value'].astype('float16').std())

       

        os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir + os.sep + os.pardir + os.sep + os.pardir))   

    os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir+os.sep + os.pardir))

    df_proc = pd.DataFrame(dic_result)

    sample_id = df_record['sample_id']
    nan_indices = df_record[df_record['measurement_id1'].isna()].index
    dic_nan = {str(i) : ['nan'] for i in df_proc.columns}
    df_nan = pd.DataFrame(dic_nan)

    for index in nan_indices:
        df_proc = pd.concat([df_proc.iloc[:index], df_nan, df_proc.iloc[index:]]).reset_index(drop=True)
    result_df = df_proc.reset_index(drop=True)

    result_df.iloc[:,0] = (sample_id).astype(str)
    cal = result_df['sample id'].str.contains('cal')
    df_cal = result_df[cal]
    df_final = result_df[np.logical_not(cal)]
    df_final.index = range(len(df_final))

    return df_final,df_cal

# Density

In [115]:
def density(drillhole):
    
    dict = pd.read_excel(f'{drillhole}/{drillhole}_sample_report.xlsx', sheet_name = None)
    df_den = dict.get('den') 
    df_out = pd.concat([df_den['sample_id'],df_den['density']],axis=1)
    df_out.rename(columns={'sample_id': 'sample id'}, inplace=True)

    return df_out

# IP

In [2]:
def ip_res (drillhole):

    data_inventory = pd.read_excel(f'{drillhole}/{drillhole}_sample_report.xlsx', sheet_name = None)
    df_record = data_inventory.get('ip_res')
    sample_id = df_record['sample_id']

    # meas_id_prop = df_record['measurement_id'].tolist()

    os.chdir(f'{drillhole}/ip_res')

    df_proc = pd.DataFrame()
    
    for dir in os.listdir():
        os.chdir(dir+'/'+'kt20_0385'+'/'+'ip')
        ip_record = pd.read_csv('ip_records.csv')
        df_proc = pd.concat([df_proc, ip_record], ignore_index=True)

        os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir+os.sep + os.pardir+os.sep + os.pardir))

    os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir + os.sep + os.pardir))


    nan_indices = df_record[df_record['measurement_id'].isna()].index

    dic_nan = {str(i) : ['nan'] for i in df_proc.columns}
    df_nan = pd.DataFrame(dic_nan)

    
    for index in nan_indices:
        df_proc = pd.concat([df_proc.iloc[:index], df_nan, df_proc.iloc[index:]]).reset_index(drop=True)
    result_df = df_proc.reset_index(drop=True)

    result_df.iloc[:,0] = (sample_id).astype(str)
    result_df.rename(columns={'Record_Id': 'sample id'}, inplace=True)
    cal = result_df['sample id'].str.contains('cal')
    df_cal = result_df[cal]
    df_final = result_df[np.logical_not(cal)]
    df_final.index = range(len(df_final))

    return df_final,df_cal

In [3]:
_,b = ip_res('TC-2531-001')

In [4]:
b

,sample id,Date,Time,M IP [mV/V],Mx [mV/V],Mx Error [%],Mx Fit [mV/V],Mx Fit Error [%],M User [mV/V],M User Error [%],...,Sample Diameter,Sample Half Core,Measure Time,Stacks,Delta T,Data Lenght,Windows count,Windows Template Name,Pro Mode,Fit calculation
0,cal_001,01.02.2023,16:22:26,180.99,105.92,16.02,98.59,23.02,84.17,27.67,...,6.3 cm,No,4,10,250 us,16000,20,Arithmetic 1s,Yes,Yes
8,cal_002,01.02.2023,17:02:58,63.63,35.54,56.67,35.19,57.04,32.13,63.02,...,4.7 cm,No,4,10,250 us,16000,20,Arithmetic 1s,Yes,Yes
12,cal_003,01.02.2023,17:26:33,193.7,111.53,17.21,109.97,17.48,95.36,20.93,...,6.3 cm,No,4,10,250 us,16000,20,Arithmetic 1s,Yes,Yes
20,cal_004,01.02.2023,18:09:7,154.58,53.7,11.36,52.47,11.7,45.95,13.38,...,4.7 cm,No,4,10,250 us,16000,20,Arithmetic 1s,Yes,Yes
32,cal_005,02.02.2023,12:19:25,103.2,49.34,41.23,48.52,42.08,43.72,47.43,...,4.7 cm,No,4,10,250 us,16000,20,Arithmetic 1s,Yes,Yes
45,cal_006,02.02.2023,14:34:41,214.24,93.31,19.54,91.6,19.95,79.95,23.34,...,4.7 cm,No,4,10,250 us,16000,20,Arithmetic 1s,Yes,Yes
56,cal_007,03.02.2023,16:27:35,380.17,129.14,15.75,125.67,16.09,101.31,19.99,...,4.7 cm,No,4,10,250 us,16000,20,Arithmetic 1s,Yes,Yes
67,cal_008,03.02.2023,17:23:45,225.07,65.99,22.2,64.16,22.64,53.63,27.05,...,4.7 cm,No,4,10,250 us,16000,20,Arithmetic 1s,Yes,Yes
78,cal_009,03.02.2023,18:18:3,73.43,28.05,59.88,27.59,60.51,24.37,68.67,...,4.7 cm,No,4,10,250 us,16000,20,Arithmetic 1s,Yes,Yes
89,cal_010,05.01.2023,18:30:51,134.9,103.63,2.34,103.09,2.43,97.39,3.32,...,6.3 cm,No,4,10,250 us,16000,20,Arithmetic 1s,Yes,Yes


# Velocity

In [362]:
def convert_csv_to_utf8(input_file):
    with open(input_file, 'rb') as f:
        enc = chardet.detect(f.read())

    if enc['encoding'].lower() == 'utf-8':
        print(f"The file {input_file} is already in UTF-8 encoding.")
    else:
        # Use a temporary DataFrame to read the CSV with the detected encoding
        df = pd.read_csv(input_file, encoding=enc['encoding'])
        
        # Save the DataFrame with UTF-8 encoding
        output_file = input_file.replace('.csv', '_utf8.csv')
        df.to_csv(output_file, encoding='utf-8', sep=',',index=False)
        print(f"File converted and saved as {output_file}")

In [373]:
def vel(drillhole,prop):

    data_inventory = pd.read_excel(f'{drillhole}/{drillhole}_sample_report.xlsx', sheet_name = None)
    df_record = data_inventory.get(prop)
    sample_id = df_record['sample_id']

    os.chdir(f'{drillhole}/{prop}')

    df_proc = pd.DataFrame()

    for dir in os.listdir():
        os.chdir(dir)

        
        convert_csv_to_utf8('Punditlink Data.csv')

        file = glob.glob('*.csv')

        rawdata = open(file[len(file)-1], 'rb').read()
        enc = chardet.detect(rawdata)
        # df = pd.read_csv(list1[0],sep='	', encoding = enc['encoding'],header = 14)

        try:
            df_data = pd.read_csv(file[len(file)-1],sep=',', encoding = enc['encoding'],header = 14)
            df_data.columns = df_data.columns.str.strip()
            record_id = df_data['ID']
            a=[]
            for i in range(1,len(df_data),25):
                a.append(i)

        except KeyError:
            df_data = pd.read_csv(file[len(file)-1],sep='\t', encoding = enc['encoding'],header = 10)
            df_data.columns = df_data.columns.str.strip()
            record_id = df_data['ID']
            a=[]
            for i in range(1,len(df_data),16):
                a.append(i)
  
        df_vel = df_data.iloc[a]
        df_proc = pd.concat([df_proc, df_vel], ignore_index=True)
        os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir))

    os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir + os.sep + os.pardir))

    nan_indices = df_record[df_record['measurement_id'].isna()].index

    dic_nan = {str(i) : ['nan'] for i in df_proc.columns}
    df_nan = pd.DataFrame(dic_nan)

    
    for index in nan_indices:
        df_proc = pd.concat([df_proc.iloc[:index], df_nan, df_proc.iloc[index:]]).reset_index(drop=True)
    result_df = df_proc.reset_index(drop=True)

 
    result_df.iloc[:,0] = (sample_id).astype(str)
    result_df.rename(columns={'ID': 'sample id'}, inplace=True)
    cal = result_df['sample id'].str.contains('cal')
    df_cal = result_df[cal]
    df_final = result_df[np.logical_not(cal)]
    df_final.index = range(len(df_final))

    return df_final

In [375]:
d = vel('TC-2531-001','v_p')

File converted and saved as Punditlink Data_utf8.csv
File converted and saved as Punditlink Data_utf8.csv
File converted and saved as Punditlink Data_utf8.csv


In [325]:
os.getcwd()

'c:\\Users\\Prithwijit\\Dropbox\\My PC (LAPTOP-7SET5QFO)\\Documents\\jupyternotebook_ws\\Database_code'

In [324]:
os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir))

# XRF

In [5]:
def extract_last_four_digits(number):
    return float(number[-4:])

In [112]:
def xrf(drillhole):
        
        os.chdir(f'{drillhole}/xrf')  

        
        df_proc = pd.DataFrame()

        for dir in os.listdir():
            os.chdir(dir)
            file = glob.glob('*.csv')
            try:
                df_xrf = pd.read_csv(file[len(file)-1],skiprows=1)
                info = df_xrf['info']
            except KeyError:
                df_xrf = pd.read_csv(file[len(file)-1])
                info = df_xrf['info']
            except KeyError:
                df_xrf = pd.read_csv(file[len(file)-1],sep=';')
                info = df_xrf['info']
            except KeyError:
                df_xrf = pd.read_csv(file[len(file)-1],sep=',')
                info = df_xrf['info']
               
                
            os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir))
            df_proc = pd.concat([df_proc, df_xrf], ignore_index=True)

        info = df_proc['info'].tolist()
        
        i = 0
        drop_list = []

        while i < len(info):
            if isinstance(info[i], str) and info[i].startswith('cal'):
                drop_list.append(i)
            if isinstance(info[i], str) and info[i].startswith('std'):
                drop_list.append(i)
            if isinstance(info[i], str) and info[i].startswith('blk'):
                drop_list.append(i)
            i += 1
        # print(drop_list)
        new_df = df_proc.drop(df_proc.index[drop_list])

        # display(new_df)

        new_df.reset_index(inplace = True)
        new_info = new_df['info'].tolist()
        sampleid_list = []
        # delete the last letter of info
        for temp_info in new_info:
            sampleid = str(temp_info)[:-1]
            sampleid_list.append(sampleid)
        # delete NaN & abcdef columns
        new_df['info']=sampleid_list
        new_df = new_df.dropna(axis=1, how='all') #drop all rows that have any NaN values
        new_df =new_df.drop( ['Instrument Serial Num','Reading #','Date','Time','Method Name','Test Label','Collimation Status','Units'],axis=1)
        # delete 'errors' columns
        new_df1=new_df.filter(regex='Error')
        new_df2=new_df.filter(regex='User')
        # new_df1=new_df.filter(regex='Compound')
        new_df = new_df.drop(new_df1,axis=1)
        new_df = new_df.drop(new_df2,axis=1)
        new_df = new_df.replace('<LOD','0.0')
        new_df = new_df.replace('na',np.NaN)
        new_df = new_df.replace('de',np.NaN)


        new_df = pd.concat([new_df.iloc[:, :-1].apply(np.float64),new_df['info']],axis=1)
        new_df['temp_info'] = new_df['info'].apply(extract_last_four_digits)

        new_df1 = new_df.groupby(['temp_info'])[['Mg Concentration','Al Concentration','Si Concentration','P Concentration','S Concentration','K Concentration','Ca Concentration','Ti Concentration','V Concentration','Cr Concentration','Mn Concentration','Fe Concentration','Mg Concentration','Co Concentration','Ni Concentration','Cu Concentration','Zn Concentration','As Concentration','Se Concentration','Rb Concentration','Sr Concentration','Y Concentration','Zr Concentration','Nb Concentration','Mo Concentration','Ag Concentration','Cd Concentration','Sn Concentration','Sb Concentration','W Concentration','Au Concentration','Hg Concentration','Pb Concentration','Bi Concentration','Th Concentration','U Concentration','LE Concentration']].mean().reset_index()
        new_df2 = new_df.groupby(['temp_info'])[['Mg Concentration','Al Concentration','Si Concentration','P Concentration','S Concentration','K Concentration','Ca Concentration','Ti Concentration','V Concentration','Cr Concentration','Mn Concentration','Fe Concentration','Mg Concentration','Co Concentration','Ni Concentration','Cu Concentration','Zn Concentration','As Concentration','Se Concentration','Rb Concentration','Sr Concentration','Y Concentration','Zr Concentration','Nb Concentration','Mo Concentration','Ag Concentration','Cd Concentration','Sn Concentration','Sb Concentration','W Concentration','Au Concentration','Hg Concentration','Pb Concentration','Bi Concentration','Th Concentration','U Concentration','LE Concentration']].std().reset_index()
        new_df3 = new_df.groupby(['temp_info'])[['Mg Concentration','Al Concentration','Si Concentration','P Concentration','S Concentration','K Concentration','Ca Concentration','Ti Concentration','V Concentration','Cr Concentration','Mn Concentration','Fe Concentration','Mg Concentration','Co Concentration','Ni Concentration','Cu Concentration','Zn Concentration','As Concentration','Se Concentration','Rb Concentration','Sr Concentration','Y Concentration','Zr Concentration','Nb Concentration','Mo Concentration','Ag Concentration','Cd Concentration','Sn Concentration','Sb Concentration','W Concentration','Au Concentration','Hg Concentration','Pb Concentration','Bi Concentration','Th Concentration','U Concentration','LE Concentration']].median().reset_index()
        new_df4 = new_df.groupby(['temp_info'])[['Mg Concentration','Al Concentration','Si Concentration','P Concentration','S Concentration','K Concentration','Ca Concentration','Ti Concentration','V Concentration','Cr Concentration','Mn Concentration','Fe Concentration','Mg Concentration','Co Concentration','Ni Concentration','Cu Concentration','Zn Concentration','As Concentration','Se Concentration','Rb Concentration','Sr Concentration','Y Concentration','Zr Concentration','Nb Concentration','Mo Concentration','Ag Concentration','Cd Concentration','Sn Concentration','Sb Concentration','W Concentration','Au Concentration','Hg Concentration','Pb Concentration','Bi Concentration','Th Concentration','U Concentration','LE Concentration']].max().reset_index()
        new_df5 = new_df.groupby(['temp_info'])[['Mg Concentration','Al Concentration','Si Concentration','P Concentration','S Concentration','K Concentration','Ca Concentration','Ti Concentration','V Concentration','Cr Concentration','Mn Concentration','Fe Concentration','Mg Concentration','Co Concentration','Ni Concentration','Cu Concentration','Zn Concentration','As Concentration','Se Concentration','Rb Concentration','Sr Concentration','Y Concentration','Zr Concentration','Nb Concentration','Mo Concentration','Ag Concentration','Cd Concentration','Sn Concentration','Sb Concentration','W Concentration','Au Concentration','Hg Concentration','Pb Concentration','Bi Concentration','Th Concentration','U Concentration','LE Concentration']].min().reset_index()
        new_df1.columns = new_df1.columns.str.replace('Concentration', 'Mean')
        # new_df1.columns = new_df1.replace('Concentration', 'Mean')
        new_df2.columns = new_df2.columns.str.replace('Concentration', 'Std')
        new_df3.columns = new_df3.columns.str.replace('Concentration', 'Median')
        new_df4.columns = new_df4.columns.str.replace('Concentration', 'Max')
        new_df5.columns = new_df5.columns.str.replace('Concentration', 'Min')
        new_df6 = pd.concat([new_df1,new_df2,new_df3,new_df4,new_df5],axis=1)
        new_df7 = new_df6.sort_index(axis=1)
        new_df7 = new_df7.loc[:, ~new_df7.columns.duplicated()]
        new_df7 = new_df7.iloc[:, :-1]
        new_df7['info']=np.unique(sampleid_list)
        df_final = new_df7 
        
        df_final.rename(columns={'info': 'sample id'}, inplace=True)

        os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir+ os.sep + os.pardir))

        return df_final

In [113]:
df1 = xrf('campaign03')

C:\Users\Prithwijit\AppData\Local\Temp\ipykernel_21440\3205387385.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  new_df.reset_index(inplace = True)


In [114]:
df1

,Ag Max,Ag Mean,Ag Median,Ag Min,Ag Std,Al Max,Al Mean,Al Median,Al Min,Al Std,...,Zn Mean,Zn Median,Zn Min,Zn Std,Zr Max,Zr Mean,Zr Median,Zr Min,Zr Std,sample id
0,0.0,0.0,0.0,0.0,0.0,1297.0,783.833333,1077.0,0.0,616.005655,...,67.333333,56.0,33.0,37.473546,8.0,4.666667,6.0,0.0,3.777124,A0192
1,0.0,0.0,0.0,0.0,0.0,969.0,453.166667,434.0,0.0,497.624122,...,58.500000,53.0,43.0,20.569395,0.0,0.000000,0.0,0.0,0.000000,A0193
2,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,...,79.500000,59.0,43.0,44.266240,0.0,0.000000,0.0,0.0,0.000000,A0194
3,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,...,119.000000,110.5,60.0,63.649038,0.0,0.000000,0.0,0.0,0.000000,A0195
4,0.0,0.0,0.0,0.0,0.0,1124.0,187.333333,0.0,0.0,458.871078,...,2653.833333,2117.5,356.0,2619.407522,0.0,0.000000,0.0,0.0,0.000000,A0196
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,0.0,0.0,0.0,0.0,0.0,924.0,280.666667,0.0,0.0,437.888875,...,62090.666667,50586.5,4216.0,59000.132561,0.0,0.000000,0.0,0.0,0.000000,A0249
58,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,...,23671.833333,13609.5,889.0,30867.641824,0.0,0.000000,0.0,0.0,0.000000,A0250
59,0.0,0.0,0.0,0.0,0.0,3980.0,1174.166667,690.5,0.0,1569.166201,...,365485.000000,380358.0,231694.0,83814.100709,0.0,0.000000,0.0,0.0,0.000000,A0251
60,0.0,0.0,0.0,0.0,0.0,3994.0,1847.833333,1721.5,0.0,1786.163309,...,112591.333333,107437.5,11300.0,93069.324434,0.0,0.000000,0.0,0.0,0.000000,A0252


In [55]:
os.getcwd()

'c:\\Users\\Prithwijit\\Dropbox\\My PC (LAPTOP-7SET5QFO)\\Documents\\jupyternotebook_ws\\Database_code'

In [54]:
os.chdir(os.path.normpath(os.getcwd() + os.sep + os.pardir+ os.sep + os.pardir))
